# 3. Markov Decision Processes for Topology Optimization

## 3.1 Introduction to MDP Framework

Markov Decision Processes (MDPs) provide a mathematical framework for modeling sequential decision-making under uncertainty. In the context of topology optimization, MDPs enable us to frame the design process as a series of intelligent modifications that progressively improve the electromagnetic performance of electrical machines.

### 3.1.1 Why MDPs for Topology Optimization?

Traditional topology optimization approaches treat design as a single-step optimization problem. The MDP formulation offers several advantages:

- **Sequential Design Process**: Mimics human engineering intuition of iterative improvement
- **Learning from Experience**: Algorithms improve through repeated design attempts
- **Partial Credit Assignment**: Intermediate design steps can be rewarded
- **Adaptive Strategy**: Design approach can evolve based on current state

### 3.1.2 Key Components of MDP

An MDP is defined by five key components:

1. **States (S)**: Current topology configuration
2. **Actions (A)**: Design modifications or decisions
3. **Transition Model (P)**: Probability of moving between states
4. **Reward Function (R)**: Performance feedback signal
5. **Policy (π)**: Strategy for selecting actions

## 3.2 Mathematical Formulation

### 3.2.1 Formal MDP Definition

A Markov Decision Process is a tuple $(S, A, P, R, \gamma)$ where:

- **$S$**: Set of possible states (topology configurations)
- **$A$**: Set of possible actions (design modifications)
- **$P(s'|s,a)$**: Transition probability from state $s$ to $s'$ given action $a$
- **$R(s,a,s')$**: Reward received when transitioning from $s$ to $s'$ via action $a$
- **$\gamma \in [0,1]$**: Discount factor for future rewards

### 3.2.2 Markov Property

The **Markov property** states that the future is independent of the past given the present:

$$P(s_{t+1}|s_t, a_t, s_{t-1}, a_{t-1}, \ldots, s_0, a_0) = P(s_{t+1}|s_t, a_t)$$

This property is crucial as it enables tractable computation and learning algorithms.

## 3.3 MDP Components in Topology Optimization

### 3.3.1 State Representation

In topology optimization, states represent the current material distribution:

**Grid-based Representation**:
$$s = \begin{bmatrix} \rho_{0,0} & \rho_{0,1} & \cdots & \rho_{0,n-1} \\ \rho_{1,0} & \rho_{1,1} & \cdots & \rho_{1,n-1} \\ \vdots & \vdots & \ddots & \vdots \\ \rho_{m-1,0} & \rho_{m-1,1} & \cdots & \rho_{m-1,n-1} \end{bmatrix}$$

Where $\rho_{i,j} \in \{0,1\}$ represents material presence (1) or absence (0) at grid position $(i,j)$.

**State Vectorization**:
For machine learning applications, the 2D grid is flattened to a 1D vector:
$$\mathbf{s} = [\rho_0, \rho_1, \ldots, \rho_{N-1}]^T$$
where $N = m \times n$ is the total number of design variables.

### 3.3.2 Action Space Design

Actions represent design modifications that transition between states:

**Single-cell Actions**:
- **Add Material**: Change $\rho_{i,j} = 0$ to $\rho_{i,j} = 1$
- **Remove Material**: Change $\rho_{i,j} = 1$ to $\rho_{i,j} = 0$
- **Toggle**: Flip the state of a single cell

**Multi-cell Actions**:
- **Add Line**: Add material along a line segment
- **Remove Region**: Clear material in a rectangular region
- **Pattern Operations**: Apply predefined patterns

**Action Encoding**:
Each action $a \in A$ can be parameterized as:
$$a = (\text{type}, i, j, \text{parameters})$$

### 3.3.3 Transition Dynamics

In topology optimization, transitions are typically **deterministic**:

$$P(s'|s,a) = \begin{cases} 1 & \text{if } s' = f(s,a) \\ 0 & \text{otherwise} \end{cases}$$

Where $f(s,a)$ is the deterministic function that applies action $a$ to state $s$.

However, transitions can be **stochastic** to model:
- Manufacturing uncertainties
- Material property variations
- Numerical noise in simulations

### 3.3.4 Reward Function Design

The reward function guides the optimization process by providing feedback on design quality:

**Performance-based Rewards**:
$$r(s,a,s') = \alpha_1 \cdot \text{Torque}(s') - \alpha_2 \cdot \text{Losses}(s') - \alpha_3 \cdot \text{Mass}(s')$$

**Improvement Rewards**:
$$r(s,a,s') = \text{Performance}(s') - \text{Performance}(s)$$

**Shaping Rewards**:
- **Intermediate Rewards**: Reward partial progress toward goals
- **Penalty Terms**: Penalize constraint violations
- **Bonus Rewards**: Extra rewards for significant improvements

## 3.4 Value Functions

Value functions quantify the long-term desirability of states or state-action pairs.

### 3.4.1 State Value Function

The **state value function** $V^\pi(s)$ represents the expected cumulative reward when starting from state $s$ and following policy $\pi$:

$$V^\pi(s) = \mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t r_t \mid s_0 = s, \pi\right]$$

### 3.4.2 Action Value Function (Q-function)

The **action value function** $Q^\pi(s,a)$ represents the expected cumulative reward when starting from state $s$, taking action $a$, and then following policy $\pi$:

$$Q^\pi(s,a) = \mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t r_t \mid s_0 = s, a_0 = a, \pi\right]$$

### 3.4.3 Optimal Value Functions

The **optimal state value function** and **optimal action value function** are:

$$V^*(s) = \max_\pi V^\pi(s) = \max_a Q^*(s,a)$$

$$Q^*(s,a) = \max_\pi Q^\pi(s,a)$$

## 3.5 Bellman Equations

Bellman equations provide recursive relationships that define optimal value functions.

### 3.5.1 Bellman Expectation Equation

For state values:
$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) [R(s,a,s') + \gamma V^\pi(s')]$$

For action values:
$$Q^\pi(s,a) = \sum_{s'} P(s'|s,a) [R(s,a,s') + \gamma \sum_{a'} \pi(a'|s') Q^\pi(s',a')]$$

### 3.5.2 Bellman Optimality Equation

For optimal state values:
$$V^*(s) = \max_a \sum_{s'} P(s'|s,a) [R(s,a,s') + \gamma V^*(s')]$$

For optimal action values:
$$Q^*(s,a) = \sum_{s'} P(s'|s,a) [R(s,a,s') + \gamma \max_{a'} Q^*(s',a')]$$

### 3.5.3 Intuitive Interpretation

The Bellman equations express that:

- **Immediate Reward**: The reward received for the current action
- **Future Value**: The discounted value of future states
- **Optimality**: Choose actions that maximize the sum of immediate and future rewards

## 3.6 Policies

A policy defines the agent's behavior by specifying action selection in each state.

### 3.6.1 Stochastic Policy

A stochastic policy $\pi: S \times A \rightarrow [0,1]$ gives the probability of taking action $a$ in state $s$:

$$\pi(a|s) = P(a_t = a | s_t = s)$$

Constraints: $\sum_a \pi(a|s) = 1$ for all $s \in S$

### 3.6.2 Deterministic Policy

A deterministic policy $\pi: S \rightarrow A$ directly maps states to actions:

$$a = \pi(s)$$

### 3.6.3 Optimal Policy

The optimal policy $\pi^*$ maximizes the expected cumulative reward:

$$\pi^* = \arg\max_\pi V^\pi(s) \quad \forall s \in S$$

## 3.7 MDP Solution Methods

Several approaches exist for solving MDPs, each with different trade-offs.

### 3.7.1 Dynamic Programming

**Value Iteration**:
$$V_{k+1}(s) = \max_a \sum_{s'} P(s'|s,a) [R(s,a,s') + \gamma V_k(s')]$$

**Policy Iteration**:
1. **Policy Evaluation**: Compute $V^\pi$ for current policy
2. **Policy Improvement**: Update $\pi$ to be greedy w.r.t. $V^\pi$
3. Repeat until convergence

### 3.7.2 Model-Free Methods

When the transition model is unknown:

- **Monte Carlo Methods**: Learn from complete episodes
- **Temporal Difference Learning**: Learn from incomplete episodes
- **Q-Learning**: Directly learn optimal action values

### 3.7.3 Function Approximation

For large state spaces, approximate value functions:

- **Linear Function Approximation**: $V(s) \approx \mathbf{w}^T \mathbf{\phi}(s)$
- **Neural Networks**: Deep RL methods
- **Policy Gradient Methods**: Direct policy optimization

## 3.8 MDPs for SynRM Design

### 3.8.1 SynRM-specific State Design

For Synchronous Reluctance Motor design, states include:

- **Flux Barrier Configuration**: Rotor topology
- **Material Distribution**: Iron vs air regions
- **Geometric Constraints**: Manufacturing limitations
- **Performance Metrics**: Current torque and efficiency

### 3.8.2 Action Space for SynRM

Specialized actions for motor design:

- **Add Flux Barrier**: Create new flux barrier
- **Modify Barrier Shape**: Change barrier geometry
- **Adjust Barrier Width**: Vary barrier thickness
- **Relocate Barrier**: Move barrier position

### 3.8.3 SynRM Reward Function

Motor-specific reward design:

$$r = w_1 \cdot \frac{T}{T_{\text{max}}} + w_2 \cdot \frac{\eta}{\eta_{\text{max}}} - w_3 \cdot \frac{T_{\text{ripple}}}{T_{\text{ripple,max}}}$$

Where:
- $T$: Average torque
- $\eta$: Efficiency
- $T_{\text{ripple}}$: Torque ripple
- $w_1, w_2, w_3$: Weighting factors

## 3.9 Practical Considerations

### 3.9.1 State Space Complexity

For an $n \times n$ grid:
- **State Space Size**: $2^{n^2}$ possible topologies
- **Curse of Dimensionality**: Exponential growth with grid size
- **Solution**: Use function approximation and hierarchical methods

### 3.9.2 Reward Shaping

Challenges in reward design:
- **Sparse Rewards**: Long delay between actions and meaningful feedback
- **Local Optima**: Policies may get stuck in suboptimal designs
- **Multi-objective Balance**: Competing design objectives

### 3.9.3 Exploration-Exploitation Trade-off

Balancing discovery of new designs with refinement of known good designs:

- **Exploration**: Try novel design modifications
- **Exploitation**: Refine promising design directions
- **Strategies**: ε-greedy, Boltzmann exploration, Upper Confidence Bounds

## 3.10 Summary

Markov Decision Processes provide a powerful framework for topology optimization by:

1. **Sequential Decision Making**: Frame design as an iterative process
2. **Learning from Experience**: Improve design strategies over time
3. **Mathematical Rigor**: Provide formal optimality guarantees
4. **Flexibility**: Adapt to different design problems and constraints

The MDP formulation enables the application of reinforcement learning algorithms to topology optimization, opening new possibilities for automated electrical machine design.

---

**Next Section**: [Q-Learning Fundamentals](04_q_learning_fundamentals.ipynb) - Implementation of RL algorithms for topology optimization.

In [ ]:
# Interactive MDP visualization for topology optimization
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, Markdown
import matplotlib.patches as mpatches

class TopologyMDP:
    """Simplified MDP for topology optimization demonstration"""
    
    def __init__(self, grid_size=4):
        self.grid_size = grid_size
        self.n_states = 2**(grid_size * grid_size)
        self.reset()
    
    def reset(self):
        """Initialize topology with some material"""
        self.state = np.zeros((self.grid_size, self.grid_size))
        # Add some initial material
        self.state[1:3, 1:3] = 1
        return self.state.copy()
    
    def step(self, action):
        """Execute action and return reward"""
        i, j = action // self.grid_size, action % self.grid_size
        
        # Toggle material at position (i,j)
        old_value = self.state[i, j]
        self.state[i, j] = 1 - self.state[i, j]
        
        # Calculate reward based on design quality
        reward = self.calculate_reward()
        
        return self.state.copy(), reward, False, {"action": (i, j), "old_value": old_value}
    
    def calculate_reward(self):
        """Simple reward function based on design metrics"""
        # Reward for having connected material
        connectivity_reward = self.measure_connectivity() * 10
        
        # Penalty for too much or too little material
        material_ratio = np.sum(self.state) / (self.grid_size * self.grid_size)
        material_penalty = -abs(material_ratio - 0.4) * 20
        
        # Bonus for symmetric designs
        symmetry_bonus = self.measure_symmetry() * 5
        
        return connectivity_reward + material_penalty + symmetry_bonus
    
    def measure_connectivity(self):
        """Measure how connected the material regions are"""
        # Simple connectivity measure (count adjacent material cells)
        connectivity = 0
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                if self.state[i, j] == 1:
                    # Check neighbors
                    for di, dj in [(0,1), (1,0), (0,-1), (-1,0)]:
                        ni, nj = i + di, j + dj
                        if 0 <= ni < self.grid_size and 0 <= nj < self.grid_size:
                            if self.state[ni, nj] == 1:
                                connectivity += 1
        return connectivity / 8  # Normalize
    
    def measure_symmetry(self):
        """Measure symmetry of the design"""
        # Horizontal symmetry
        h_sym = np.mean(np.abs(self.state - np.fliplr(self.state)))
        # Vertical symmetry  
        v_sym = np.mean(np.abs(self.state - np.flipud(self.state)))
        # Return symmetry score (lower difference = higher symmetry)
        return 2 - (h_sym + v_sym)

# Demonstrate MDP dynamics
mdp = TopologyMDP(grid_size=6)
state = mdp.reset()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Simulate sequence of actions
actions_sequence = [10, 15, 20, 25, 8, 18]
rewards = []

for idx, action in enumerate(actions_sequence):
    if idx > 0:
        state, reward, _, info = mdp.step(action)
        rewards.append(reward)
    
    # Visualize current state
    ax = axes[idx]
    im = ax.imshow(state, cmap='binary', interpolation='nearest')
    ax.set_title(f'Step {idx}: State\nAction: {action} → {info.get("action", "N/A")}')
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')
    ax.grid(True, alpha=0.3)
    
    # Add reward text
    if idx > 0:
        reward_text = f'Reward: {reward:.2f}'
        if reward > 0:
            reward_text += ' ✓'
            color = 'green'
        else:
            reward_text += ' ✗'
            color = 'red'
        ax.text(0.02, 0.98, reward_text, transform=ax.transAxes, 
                fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))

plt.suptitle('MDP Dynamics in Topology Optimization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Show cumulative reward
cumulative_rewards = np.cumsum(rewards)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(rewards)+1), rewards, 'bo-', label='Immediate Reward')
ax.plot(range(1, len(rewards)+1), cumulative_rewards, 'r^-', label='Cumulative Reward')
ax.set_xlabel('Step')
ax.set_ylabel('Reward')
ax.set_title('Reward Evolution in Topology Optimization MDP')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

display(MMarkdown("""
**Figure: MDP Demonstration for Topology Optimization**

This visualization shows how an MDP framework works for topology optimization:

1. **State Representation**: Each grid shows the current material distribution (black = material, white = void)
2. **Actions**: Each step toggles material at a specific location
3. **Rewards**: The reward function evaluates design quality based on:
   - Connectivity of material regions
   - Appropriate material ratio (40% target)
   - Symmetry of the design

4. **Learning**: The agent learns which actions lead to better designs through reward feedback

Key insights from this MDP formulation:
- Each action has immediate and long-term consequences
- The reward function guides the optimization process
- Sequential decision-making enables progressive design improvement
"""))